# Transfer encoder SIBI ke classifier BISINDO baru

Pilih **Runtime ? Change runtime type ? GPU**. Upload paket `bisindo-transfer-v1-bisindo-colab.zip` yang sudah disiapkan lokal.

Tahap pertama hanya melatih classifier baru; encoder SIBI dibekukan. Tahap kedua melatih encoder dan classifier dengan learning rate kecil. Checkpoint SIBI sumber tetap utuh. Semua output adalah checkpoint BISINDO terpisah. Tidak membaca array test atau foto sumber.


In [ ]:
%pip install -q "scikit-learn==1.5.2" "joblib==1.4.2" "matplotlib>=3.8,<4"
import torch
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU fallback; aktifkan GPU bila diinginkan')


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, os
uploaded = files.upload()  # bisindo-transfer-v1-bisindo-colab.zip
if len(uploaded) != 1:
    raise ValueError('Upload satu paket ZIP koordinat saja')
project = Path('/content/coordinate-project')
project.mkdir(exist_ok=False)
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as archive:
    for item in archive.infolist():
        target = (project / item.filename).resolve()
        if not target.is_relative_to(project.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(project)
os.chdir(project)
print('Project:', project)


In [ ]:
LANGUAGE = 'bisindo'
HEAD_EPOCHS = 10
FINETUNE_EPOCHS = 60
RUN_DIR = 'artifacts/bisindo-transfer-v1/bisindo_colab'

import subprocess, sys
command = [sys.executable, '-u', 'scripts/train_bisindo_transfer.py',
           '--config', 'configs/bisindo_transfer_v1.json',
           '--features-dir', 'data/features/coordinate-reset-v1/bisindo',
           '--source-run', 'artifacts/coordinate-reset-v1/sibi_20261020_130607_889055',
           '--head-epochs', str(HEAD_EPOCHS), '--finetune-epochs', str(FINETUNE_EPOCHS),
           '--output-dir', RUN_DIR]
subprocess.run(command, check=True)


In [ ]:
import json
from IPython.display import display, Image
selection = json.loads((Path(RUN_DIR) / 'selection.json').read_text())
print('Selected:', selection['selected'])
print('Test loaded:', selection['test_loaded'])
display(Image(filename=str(Path(RUN_DIR) / 'learning_curves.png')))
# Detail per kelas/sumber ada di *_validation.json; epoch ada di *_epochs.csv.


In [ ]:
import zipfile
archive_path = Path('/content') / f'{LANGUAGE}-transfer-results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in Path(RUN_DIR).rglob('*'):
        if path.is_file():
            archive.write(path, path.as_posix())
files.download(str(archive_path))


## Kembali ke komputer lokal

Ekstrak ZIP hasil ke akar proyek dengan mempertahankan folder `artifacts/`. Jalankan:

```powershell
python scripts/webcam_coordinates.py --run-dir artifacts/bisindo-transfer-v1/bisindo_colab --check-only
python scripts/webcam_coordinates.py --run-dir artifacts/bisindo-transfer-v1/bisindo_colab
```

Gunakan environment MediaPipe lokal lama untuk webcam. Skor Softmax/margin SVM belum terkalibrasi; performa validation tidak menjamin performa webcam. Source notebook ini tidak menyimpan output training. Akurasi classifier-only dan sistem berbeda karena gambar gagal ekstraksi tetap dihitung sebagai penolakan.
